In [1]:
import re

def parse(expression):
    tokens = re.findall(r'[A-Za-z_][A-Za-z0-9_]*|[(),]', expression)
    pos = 0

    def term():
        nonlocal pos

        name = tokens[pos]
        pos += 1

        if pos < len(tokens) and tokens[pos] == '(':
            pos += 1
            args = []

            while True:
                args.append(term())

                if tokens[pos] == ')':
                    pos += 1
                    break

                pos += 1

            return (name, args)

        return name

    return term()


def apply_substitution(term, subst):
    if isinstance(term, str):
        if term in subst:
            return apply_substitution(subst[term], subst)
        return term

    name, args = term
    return (name, [apply_substitution(a, subst) for a in args])


def occurs(variable, term, subst):
    term = apply_substitution(term, subst)

    if variable == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(variable, a, subst) for a in term[1])

    return False


def unify(t1, t2, subst):

    t1 = apply_substitution(t1, subst)
    t2 = apply_substitution(t2, subst)

    if t1 == t2:
        return subst

    if isinstance(t1, str) and t1[0].islower():
        if occurs(t1, t2, subst):
            return None

        subst[t1] = t2
        return subst

    if isinstance(t2, str) and t2[0].islower():
        if occurs(t2, t1, subst):
            return None

        subst[t2] = t1
        return subst

    if isinstance(t1, tuple) and isinstance(t2, tuple):

        if t1[0] != t2[0]:
            return None

        if len(t1[1]) != len(t2[1]):
            return None

        for a, b in zip(t1[1], t2[1]):
            subst = unify(a, b, subst)

            if subst is None:
                return None

        return subst

    return None


print("UNIFICATION ALGORITHM")

e1 = input("Enter first expression: ")
e2 = input("Enter second expression: ")

t1 = parse(e1)
t2 = parse(e2)

result = unify(t1, t2, {})

if result is None:
    print("\nUnification Failed!")
else:
    print("\nUnification Successful!")
    print("Substitution Set:")

    for key, value in result.items():
        value = apply_substitution(value, result)
        print(key, "=", value)

UNIFICATION ALGORITHM
Enter first expression: P(x, John)
Enter second expression:  P(Mary, y)

Unification Successful!
Substitution Set:
x = Mary
y = John
